# Ders 18 — Aynı görev, dört komut: clean / empty / wrong / tr × 10 init state (T4)

**Önce:** Çalışma zamanı türü **T4 GPU** (Kaggle: Accelerator **GPU T4 x2**, internet açık; `/content` → `/kaggle/working`). Ders 12'nin runtime'ı açıksa hücre 1–5'i atlayıp 6'dan devam edebilirsin (orijinal LIBERO kurulu olmalı; LIBERO-Plus runtime'ı DEĞİL — Ders 16 uyarısı).

Akış: (1) donanım → (2) EGL → (3) VLANeXt + LIBERO → (4) Ders 12 yaması → (5) checkpoint → **(6) Ders 18 yaması** → (7) koşullar + config → (8) tek koşu: 1 model yüklemesi + 40 episode → (9) analiz: tablo, `episodes.csv`, `runs.csv` → (10) videolar.

Süre (hipotez, T4): kurulum ~10 dk + indirme ~5 dk + model yükleme ~3 dk + 40 episode ~30–60 dk (başarısız episode 220 adıma kadar sürer; `empty`/`tr` koşulları daha uzun olabilir). Ölçtüğün süreleri `projects/05-…/README.md`'ye yaz.

In [ ]:
# 1) Donanım: cc (7, 5) = T4
!nvidia-smi --query-gpu=name,memory.total --format=csv
import torch, psutil, shutil
print("torch", torch.__version__, "| GPU", torch.cuda.get_device_name(0), "| cc", torch.cuda.get_device_capability(0))
print("RAM %.1f GB | disk boş %.0f GB" % (psutil.virtual_memory().total/1e9, shutil.disk_usage("/").free/1e9))

In [ ]:
# 2) MuJoCo EGL (VLANeXt COMMON_ISSUES #1)
!apt-get -qq update && apt-get -qq install -y libgl1-mesa-dev libegl1-mesa-dev libxrandr2 libxcursor1 libxinerama1 libxrender1 > /dev/null
import os; os.environ["MUJOCO_GL"] = "egl"; os.environ["PYOPENGL_PLATFORM"] = "egl"; print("ok")

In [ ]:
# 3) VLANeXt + orijinal LIBERO (Ders 12 hücre 3 ile aynı)
%cd /content
!git clone -q https://github.com/DravenALG/VLANeXt && echo "clone ok"
%cd /content/VLANeXt
!git log -1 --format='VLANeXt %h %ci'
!mkdir -p third_party && git clone -q https://github.com/Lifelong-Robot-Learning/LIBERO.git third_party/LIBERO
!pip install -q "transformers==5.5.0" "diffusers==0.36.0" accelerate huggingface_hub sentencepiece h5py opencv-python "imageio[ffmpeg]" \
    "robosuite==1.4.1" "mujoco==3.8.1" bddl easydict cloudpickle gym pyyaml wandb tqdm tensorflow-datasets rlds scipy 2>&1 | tail -2
!pip install -q ./third_party/LIBERO 2>&1 | tail -1
!printf 'N\n' | python -c "import libero.libero; print('LIBERO ok')"

In [ ]:
# 4) Ders 12 yaması (sdpa, mmap, VLANEXT_TASK_IDS) — Ders 18 yamasından ÖNCE
!curl -sO https://raw.githubusercontent.com/mesutd0nmez/vla-robot-learning-tr/main/code/day12/vlanext_t4_patch.py
!python vlanext_t4_patch.py /content/VLANeXt

In [ ]:
# 5) Checkpoint (16.4 GB; Ders 12/16 ile aynı dosya)
!hf download DravenALG/VLANeXt VLANeXt_libero_spatial.pt --local-dir checkpoints && ls -lh checkpoints

In [ ]:
# 6) Ders 18 yaması: koşul = EpisodeSpec alanı; init state × koşul; log'a condition=; video adına koşul; RUN_TAG
!curl -sO https://raw.githubusercontent.com/mesutd0nmez/vla-robot-learning-tr/main/code/day18/vlanext_cond_patch.py
!python vlanext_cond_patch.py /content/VLANeXt
# Beklenen: 10 "[+] uygulandı" (ilk kez) — "HATA: beklenen satır bulunamadı" görürsen repo değişmiştir; hücre 8b'deki B planına geç.

In [ ]:
# 7) Koşullar (datasets/01'den üretilmiş JSON) + config. clean = null → eval kodu task.language'ı kendisi koyar.
!curl -sO https://raw.githubusercontent.com/mesutd0nmez/vla-robot-learning-tr/main/code/day18/conditions_task0.json && curl -s -o config/t4_cond_config.yaml https://raw.githubusercontent.com/mesutd0nmez/vla-robot-learning-tr/main/code/day18/t4_cond_config.yaml
import json; CONDS = open("conditions_task0.json", encoding="utf-8").read(); print(json.dumps(json.loads(CONDS), ensure_ascii=False, indent=2))
# Kaggle'daysan (29 GB RAM): num_parallel_envs 2 → 4, böylece aynı init state'in dört koşulu aynı batch'te koşar
# !sed -i 's/num_parallel_envs: 2/num_parallel_envs: 4/' config/t4_cond_config.yaml

In [ ]:
# 8) TEK koşu: görev 0, 10 init state × 4 koşul = 40 episode, bir model yüklemesi. (~35–65 dk hipotez)
import os, time
os.environ["VLANEXT_TASK_IDS"] = "0"
os.environ["VLANEXT_CONDITIONS"] = CONDS
os.environ["VLANEXT_RUN_TAG"] = "_4cond"
os.environ["PYTHONPATH"] = "/content/VLANeXt/third_party/LIBERO"
os.environ["MUJOCO_EGL_DEVICE_ID"] = "0"
os.environ.pop("LIBERO_CONFIG_PATH", None); os.environ.pop("VLANEXT_INSTR_OVERRIDE", None)
t0 = time.time()
!CUDA_VISIBLE_DEVICES=0 python -m scripts.libero_bench_eval --config config/t4_cond_config.yaml 2>&1 | grep -v Warning | grep -E "Conditions|Eval setup|Episode result|Eval finished|Error|error" | tee eval_stdout_4cond.txt
print("toplam süre %.1f dk" % ((time.time() - t0) / 60))

### 8b) B planı — yalnızca Ders 18 yaması uygulanamadıysa
Ders 16'nın yamasını (`VLANEXT_INSTR_OVERRIDE`) kullanıp dört ayrı koşu yap (4 model yüklemesi; aynı `seed`, aynı 10 init state):
```python
!curl -sO https://raw.githubusercontent.com/mesutd0nmez/vla-robot-learning-tr/main/code/day16/vlanext_plus_patch.py && python vlanext_plus_patch.py /content/VLANeXt
import subprocess, glob
for cond, text in [("clean", None), ("empty", ""), ("wrong", json.loads(CONDS)["wrong"]), ("tr", json.loads(CONDS)["tr"])]:
    env = dict(os.environ, VLANEXT_TASK_IDS="0", VLANEXT_RUN_TAG="_" + cond); env.pop("VLANEXT_CONDITIONS", None); env.pop("VLANEXT_INSTR_OVERRIDE", None)
    if text is not None: env["VLANEXT_INSTR_OVERRIDE"] = text
    subprocess.run(["python", "-m", "scripts.libero_bench_eval", "--config", "config/t4_cond_config.yaml"], env=env, stdout=open(f"stdout_{cond}.txt", "w"), stderr=subprocess.STDOUT)
    print(cond, sorted(glob.glob(f"checkpoints/libero_spatial_checkpoint_*_libero_{cond}*/eval.log")))
```
Not: Ders 16 yaması `libero_plus_bench_eval.py`'ye RUN_TAG ekler, `libero_bench_eval.py`'ye eklemez; RUN_TAG için Ders 18 yamasının 5. maddesi gerekir — o da uygulanamadıysa her koşudan sonra `checkpoints/libero_spatial_checkpoint_*_libero_SR*` klasörünü elle yeniden adlandır. Analiz: `python analyze_4cond.py --logs clean=... empty=... wrong=... tr=... --out results/`.

In [ ]:
# 9) Analiz: tablo (k/n, Wilson %95, uyumsuz çiftler, McNemar / Fisher p) + episodes.csv + runs.csv
!curl -sO https://raw.githubusercontent.com/mesutd0nmez/vla-robot-learning-tr/main/code/day18/analyze_4cond.py
import glob
logs = sorted(glob.glob("checkpoints/libero_spatial_checkpoint_*_libero_4cond*/eval.log"), key=os.path.getmtime)
print(logs)
!python analyze_4cond.py {logs[-1]} --run-id vlanext_sp0_4cond_01 --out results/
!cat results/episodes.csv | head -12

In [ ]:
# 10) Videolar: her koşuldan bir başarısız episode izle — gripper HANGİ kâseye gidiyor? (wrong koşulunda 'faithful' mı 'biased' mı?)
from IPython.display import Video, display
vids = sorted(glob.glob("checkpoints/libero_spatial_checkpoint_*_libero_4cond*/*.mp4"))
for cond in ["clean", "empty", "wrong", "tr"]:
    fails = [v for v in vids if f"task={cond}__" in v and "success=False" in v]
    print(cond, "başarısız:", len(fails), "/", len([v for v in vids if f"task={cond}__" in v]))
    if fails: display(Video(fails[0], embed=True, width=320))

## Sonra
- `results/` (episodes.csv, runs.csv, table.md), `eval.log` ve koşul başına bir başarısız video → `projects/05-spatial0-4kosul/` (şablondan aç; README'ye hücre süreleri, episode başına ortalama süre, `wrong` koşulunda kaç videoda öteki kâseye gidildiği).
- `runs.csv`'nin 4 satırı repo kuralına uygundur (`run_id, model, checkpoint, suite, task_id, lang, condition, n_episodes, success_rate`).
- Ders 19: bu tabloyu okuyacağız — hangi örüntü (görsel 07), shortcut var mı, Wilson aralıkları ne diyor, 20/50 episode'a çıkmaya değer mi.